In [0]:
catalog = "flights_project"

df_bronze = spark.table(f"{catalog}.bronze.flights_raw")
display(df_bronze.limit(5))

YEAR,MONTH,DAY_OF_MONTH,FL_DATE,OP_UNIQUE_CARRIER,TAIL_NUM,OP_CARRIER_FL_NUM,ORIGIN_AIRPORT_ID,ORIGIN_AIRPORT_SEQ_ID,ORIGIN_CITY_MARKET_ID,ORIGIN,ORIGIN_CITY_NAME,ORIGIN_STATE_ABR,DEST_AIRPORT_ID,DEST_AIRPORT_SEQ_ID,DEST_CITY_MARKET_ID,DEST,DEST_CITY_NAME,DEST_STATE_ABR,CRS_DEP_TIME,DEP_TIME,DEP_DELAY,DEP_DELAY_NEW,CRS_ARR_TIME,ARR_TIME,ARR_DELAY,ARR_DELAY_NEW,CANCELLED,CANCELLATION_CODE,DIVERTED,ACTUAL_ELAPSED_TIME,AIR_TIME,DISTANCE,CARRIER_DELAY,WEATHER_DELAY,NAS_DELAY,SECURITY_DELAY,LATE_AIRCRAFT_DELAY,_rescued_data,_ingested_at,_source_file
2025,7,16,7/16/2025 12:00:00 AM,NK,N612NK,1406,13796,1379611,32457,OAK,"Oakland, CA",CA,12889,1288904,32211,LAS,"Las Vegas, NV",NV,923,913,-10.0,0.0,1059,1046,-13.0,0.0,0.0,null,0.0,93.0,73.0,407.0,null,null,null,null,null,null,2026-08-16T18:50:54.883Z,/Volumes/flights_project/bronze/raw_landing/flights_2025_07.csv
2025,7,16,7/16/2025 12:00:00 AM,NK,N612NK,1411,12889,1288904,32211,LAS,"Las Vegas, NV",NV,10800,1080003,32575,BUR,"Burbank, CA",CA,1154,1145,-9.0,0.0,1306,1253,-13.0,0.0,0.0,null,0.0,68.0,44.0,223.0,null,null,null,null,null,null,2026-08-16T18:50:54.883Z,/Volumes/flights_project/bronze/raw_landing/flights_2025_07.csv
2025,7,16,7/16/2025 12:00:00 AM,NK,N612NK,1446,12889,1288904,32211,LAS,"Las Vegas, NV",NV,14908,1490804,32575,SNA,"Santa Ana, CA",CA,1752,1756,4.0,4.0,1903,1858,-5.0,0.0,0.0,null,0.0,62.0,41.0,226.0,null,null,null,null,null,null,2026-08-16T18:50:54.883Z,/Volumes/flights_project/bronze/raw_landing/flights_2025_07.csv
2025,7,16,7/16/2025 12:00:00 AM,NK,N612NK,1447,14908,1490804,32575,SNA,"Santa Ana, CA",CA,12889,1288904,32211,LAS,"Las Vegas, NV",NV,1958,1946,-12.0,0.0,2114,2053,-21.0,0.0,0.0,null,0.0,67.0,46.0,226.0,null,null,null,null,null,null,2026-08-16T18:50:54.883Z,/Volumes/flights_project/bronze/raw_landing/flights_2025_07.csv
2025,7,16,7/16/2025 12:00:00 AM,NK,N612NK,1674,12889,1288904,32211,LAS,"Las Vegas, NV",NV,10821,1082106,30852,BWI,"Baltimore, MD",MD,2248,2243,-5.0,0.0,605,554,-11.0,0.0,0.0,null,0.0,251.0,232.0,2106.0,null,null,null,null,null,null,2026-08-16T18:50:54.883Z,/Volumes/flights_project/bronze/raw_landing/flights_2025_07.csv


In [0]:
from pyspark.sql.functions import to_date, col

df_typed = (df_bronze
    .withColumn("flight_date", to_date(col("FL_DATE"), "M/d/yyyy h:mm:ss a"))
    .withColumn("dep_delay", col("DEP_DELAY").cast("double"))
    .withColumn("arr_delay", col("ARR_DELAY").cast("double"))
    .withColumn("distance", col("DISTANCE").cast("double"))
    .withColumn("air_time", col("AIR_TIME").cast("double"))
    .withColumn("carrier_delay", col("CARRIER_DELAY").cast("double"))
    .withColumn("weather_delay", col("WEATHER_DELAY").cast("double"))
    .withColumn("nas_delay", col("NAS_DELAY").cast("double"))
    .withColumn("security_delay", col("SECURITY_DELAY").cast("double"))
    .withColumn("late_aircraft_delay", col("LATE_AIRCRAFT_DELAY").cast("double"))
)

In [0]:
from pyspark.sql.functions import when, col

df_status = df_typed.withColumn(
    "flight_status",
    when(col("CANCELLED") == 1, "CANCELLED")
    .when(col("DIVERTED") == 1, "DIVERTED")
    .when(col("dep_delay") >= 15, "DELAYED")
    .otherwise("ON_TIME")
)

In [0]:
from pyspark.sql.functions import dayofweek, hour, to_timestamp, when as when2, col

df_derived = (df_status
    .withColumn("dep_hour", (col("CRS_DEP_TIME") / 100).cast("int"))
    .withColumn("day_of_week", dayofweek(col("flight_date")))
    .withColumn("is_weekend", when2(dayofweek(col("flight_date")).isin(1, 7), True).otherwise(False))
)

In [0]:
df_silver = (df_derived
    .dropDuplicates(["flight_date", "OP_UNIQUE_CARRIER", "OP_CARRIER_FL_NUM", "ORIGIN", "DEST"])
    .select(
        "flight_date", "OP_UNIQUE_CARRIER", "TAIL_NUM", "OP_CARRIER_FL_NUM",
        "ORIGIN", "ORIGIN_CITY_NAME", "ORIGIN_STATE_ABR",
        "DEST", "DEST_CITY_NAME", "DEST_STATE_ABR",
        "dep_hour", "day_of_week", "is_weekend",
        "dep_delay", "arr_delay", "flight_status",
        "distance", "air_time",
        "carrier_delay", "weather_delay", "nas_delay", "security_delay", "late_aircraft_delay",
        "_ingested_at", "_source_file"
    )
)

In [0]:
(df_silver.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{catalog}.silver.flights_clean"))

In [0]:
%sql
SELECT COUNT(*) AS row_count FROM flights_project.silver.flights_clean;

SELECT flight_status, COUNT(*) AS cnt
FROM flights_project.silver.flights_clean
GROUP BY flight_status
ORDER BY cnt DESC;

flight_status,cnt
ON_TIME,1818973
DELAYED,495407
CANCELLED,39903
DIVERTED,6686
